In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    candidate_str = str(candidate)
    if candidate.exists() and candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

In [ ]:
import os 
import threading
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
 
 

from src.drive_service.auth_service import load_creds
from src.drive_service.fs_utils import ensure_dir
from src.drive_service.index import MapIndex
from src.drive_service.index_runtime import resolve_output_path, update_index_meta
from src.drive_service.io_json import write_json
from src.drive_service.logging_utils import get_logger, setup_logging
from src.drive_service.schema import IndexFile

from src.extract_text_from_index.options import ExtractTextFromIndexOptions
from src.extract_text_from_index.planning import build_initial_stats, collect_docs
from src.extract_text_from_index.runtime import _apply_result, _flush_progress
from src.extract_text_from_index.workers import download_pdf_bytes, extract_and_write

  

In [ ]:
from src.pipeline_paths import build_pipelines_paths

root = "1hxSSsFQNMo63L_dDEbyNtVTa1I7IUjQC"
paths = build_pipelines_paths(root)

if not paths.scan_output.exists():
    raise FileNotFoundError(f"Scan output directory does not exist: {paths.scan_output}")

paths.scan_output, paths.root_output


In [ ]:
 

options = ExtractTextFromIndexOptions(
    index= paths.scan_output / "included_index.json",
    out=paths.text_extraction_output,
    included="included.index.json",
    excluded="excluded.index.json",
    report="extract_text_from_index.report.json",
    workers=8,
    extract_workers=max(1, os.cpu_count() or 1),
    max_in_flight=128,
    flush_every=100,
    log_every=50,
    skip_included=True,
    reprocess_included=False,
    reprocess_excluded=False,
    verbose=True,
)
options


In [ ]:
setup_logging(options.verbose)
logger = get_logger()

ensure_dir(options.out)
index_path = options.index if os.path.isabs(options.index) else os.path.abspath(options.index)
included_path = resolve_output_path(options.out, options.included)
excluded_path = resolve_output_path(options.out, options.excluded)
report_path = resolve_output_path(options.out, options.report)

skip_included = options.skip_included and not options.reprocess_included
skip_excluded = not options.reprocess_excluded
download_workers = max(
    1,
    options.download_workers if options.download_workers is not None else options.workers,
)
extract_workers = max(1, options.extract_workers)
max_in_flight = max(1, options.max_in_flight)
flush_every = max(1, options.flush_every)
log_every = max(1, options.log_every)

Path(index_path), Path(included_path), Path(excluded_path), Path(report_path)


In [ ]:
source = MapIndex.load_index(index_path, strict=True)
existing_included = MapIndex.load_index(included_path, strict=False)
existing_excluded = MapIndex.load_index(excluded_path, strict=False)

included_map: dict[str, IndexFile] = dict(existing_included.files)
excluded_map: dict[str, IndexFile] = dict(existing_excluded.files)
stats = build_initial_stats(len(source.files))

docs = collect_docs(
    source.files,
    included_map,
    excluded_map,
    skip_included=skip_included,
    skip_excluded=skip_excluded,
    limit=options.limit,
    stats=stats,
)

included_index = MapIndex.generate_index(source.root_id, source.employee_count, included_map)
excluded_index = MapIndex.generate_index(source.root_id, source.employee_count, excluded_map)

logger.info("Queued %s files for extraction", stats["queued"])
stats, docs[:3]


In [ ]:
items: list[dict] = []
t0 = time.time()
processed = 0
stop_event = threading.Event()

if docs:
    logger.info("Loading credentials...")
    creds = load_creds()
    logger.info("Credentials loaded")

    extract_futures = {}
    with ThreadPoolExecutor(max_workers=download_workers) as download_pool, ProcessPoolExecutor(
        max_workers=extract_workers
    ) as extract_pool:
        download_futures = [
            download_pool.submit(download_pdf_bytes, creds, doc, stop_event)
            for doc in docs
        ]
        try:
            for future in as_completed(download_futures):
                try:
                    download_result = future.result()
                except Exception as exc:
                    download_result = {
                        "status": "failed",
                        "stage": "download",
                        "reason": f"{type(exc).__name__}: {exc}",
                        "doc": {},
                    }

                if download_result["status"] != "success":
                    processed += 1
                    stats["processed"] = processed
                    _apply_result(download_result, stats, items, included_map, excluded_map)
                    _flush_progress(
                        processed=processed,
                        flush_every=flush_every,
                        log_every=log_every,
                        start_ts=t0,
                        included_index=included_index,
                        excluded_index=excluded_index,
                        included_map=included_map,
                        excluded_map=excluded_map,
                        included_path=included_path,
                        excluded_path=excluded_path,
                        logger=logger,
                    )
                    continue

                while len(extract_futures) >= max_in_flight:
                    done = next(as_completed(extract_futures))
                    doc_info = extract_futures.pop(done, {})
                    try:
                        result = done.result()
                    except Exception as exc:
                        result = {
                            "status": "failed",
                            "stage": "extract",
                            "reason": f"{type(exc).__name__}: {exc}",
                            "doc": doc_info,
                        }
                    processed += 1
                    stats["processed"] = processed
                    _apply_result(result, stats, items, included_map, excluded_map)
                    _flush_progress(
                        processed=processed,
                        flush_every=flush_every,
                        log_every=log_every,
                        start_ts=t0,
                        included_index=included_index,
                        excluded_index=excluded_index,
                        included_map=included_map,
                        excluded_map=excluded_map,
                        included_path=included_path,
                        excluded_path=excluded_path,
                        logger=logger,
                    )

                doc_info = download_result["doc"]
                extract_future = extract_pool.submit(
                    extract_and_write,
                    download_result["data"],
                    doc_info,
                    options.out,
                    options.min_normal_score,
                    options.min_score_delta,
                )
                extract_futures[extract_future] = doc_info

            for done in as_completed(list(extract_futures)):
                doc_info = extract_futures.pop(done, {})
                try:
                    result = done.result()
                except Exception as exc:
                    result = {
                        "status": "failed",
                        "stage": "extract",
                        "reason": f"{type(exc).__name__}: {exc}",
                        "doc": doc_info,
                    }
                processed += 1
                stats["processed"] = processed
                _apply_result(result, stats, items, included_map, excluded_map)
                _flush_progress(
                    processed=processed,
                    flush_every=flush_every,
                    log_every=log_every,
                    start_ts=t0,
                    included_index=included_index,
                    excluded_index=excluded_index,
                    included_map=included_map,
                    excluded_map=excluded_map,
                    included_path=included_path,
                    excluded_path=excluded_path,
                    logger=logger,
                )
        except KeyboardInterrupt:
            stop_event.set()
            logger.warning("Interrupted by user, flushing indexes...")
else:
    logger.info("No files queued")

stats


In [ ]:
included_index.files = dict(included_map)
excluded_index.files = dict(excluded_map)
update_index_meta(included_index)
update_index_meta(excluded_index)

included_index.save_index(included_path)
excluded_index.save_index(excluded_path)

Path(included_path), Path(excluded_path)


In [ ]:
payload = {
    "source_index": index_path,
    "included_index": included_path,
    "excluded_index": excluded_path,
    "out_dir": str(options.out),
    "stats": stats,
    "items": items,
    "duration_s": round(time.time() - t0, 3),
}

ensure_dir(os.path.dirname(report_path) or ".")
write_json(report_path, payload)
logger.info("Report saved to %s", report_path)

payload["stats"]
